# Lab 3.1 — Decision Trees vs. an LLM

*Chapter 3 — Machine Learning Essentials · 45 minutes · JupyterLab + scikit-learn + pandas + the OpenAI API via `course_ai`*

Not every prediction needs a language model. You will grow a small decision tree
on a synthetic code-review dataset — interpretable, auditable, free per
prediction — then ask an LLM to make the same predictions from the same rows and
compare accuracy, interpretability, latency, and cost. The point is not "trees
beat LLMs". It is: **match the tool to the problem shape**, and have evidence
for the match.

## Objectives

By the end of this lab, you will:

- Train a small `DecisionTreeClassifier` on tabular data and *read* what it
  learned — importances, rules, depth.
- Drive the same decision with an LLM from a CSV-rendered prompt, and compare
  predictions row by row.
- Argue tool choice from evidence: accuracy, interpretability, latency, cost.

## Setup

- **Libraries:** `scikit-learn` and `pandas` (pre-installed on the VM; elsewhere
  `pip install scikit-learn pandas`).
- **Key:** `OPENAI_API_KEY` from the environment / course `.env`, never printed.
  `COURSE_AI_MOCK=1` (or no key) engages mock mode: the LLM step then uses a
  fixed prediction list (8/10 correct by design), so the comparison runs offline.
- **Data:** synthetic, generated below with a fixed seed — identical every run.

In [ ]:
import re
import time

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

import course_ai
from course_ai import chat

print("mode:", course_ai.mode())
print("sklearn:", sklearn.__version__, "| pandas:", pd.__version__)

## Steps

### Step 1 — A synthetic code-review dataset (8 min)

400 code changes, five features each, and a **planted rule** for whether the
review passes — small, tested changes by non-rookies pass — plus 8% random
flips, because real review data has noise. The question for the rest of the lab:
can each tool *recover* the rule?

In [ ]:
rng = np.random.RandomState(42)
N = 400
df = pd.DataFrame({
    "lines_changed": rng.randint(5, 600, N),
    "files_touched": rng.randint(1, 16, N),
    "author_tenure": np.round(rng.uniform(0.25, 10.0, N), 1),   # years on the team
    "has_tests": rng.binomial(1, 0.6, N),
    "weekday": rng.randint(0, 5, N),                            # 0=Mon .. 4=Fri
})

# The planted rule (what both tools should try to recover):
rule = ((df["lines_changed"] <= 250) & (df["files_touched"] <= 8)
        & ((df["has_tests"] == 1) | (df["author_tenure"] >= 2.0)))
y = rule.astype(int).to_numpy().copy()
flip = rng.rand(N) < 0.08          # 8% of reviews defy the rule (human noise)
y[flip] = 1 - y[flip]
df["review_passes"] = y

FEATURES = ["lines_changed", "files_touched", "author_tenure", "has_tests", "weekday"]
print(df.head(8).to_string())
print(f"\n{N} changes | overall pass rate {df['review_passes'].mean():.0%}")

### Step 2 — Train a small decision tree (8 min)

Train/validation split, then a `DecisionTreeClassifier` small enough to *read*.
`max_depth` is the whole bias/variance story in one knob: too shallow misses the
rule, too deep memorizes the noise. Pick a depth, train, and check validation
accuracy — then try a couple of others.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    df[FEATURES], df["review_passes"], test_size=0.25, random_state=7)

tree = None
# YOUR CODE: train a small tree, e.g.
#   tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)
# Then re-run with max_depth=2 and max_depth=12 — what happens to train vs val accuracy?
if tree is None:
    tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)
    print("(reference tree trained at max_depth=4 — write your own above)\n")

train_acc = tree.score(X_train, y_train)
val_acc = tree.score(X_val, y_val)
print(f"train accuracy {train_acc:.2f} | validation accuracy {val_acc:.2f} "
      f"| depth {tree.get_depth()}")

### Step 3 — Read the model (7 min)

A decision tree is not a black box. Feature importances rank what it splits on;
`export_text` prints the actual rules. Compare what it learned with the rule you
planted — where does it agree, and where did the 8% noise lead it astray?

![A decision tree splits on feature thresholds until the leaves are homogeneous — that readability is the point](diagrams/ch03_decision_tree.png)

*A decision tree splits on feature thresholds until the leaves are homogeneous — that readability is the point (Chapter 3 deck).*

In [ ]:
print("feature importances:")
for name, score in sorted(zip(FEATURES, tree.feature_importances_), key=lambda t: -t[1]):
    print(f"  {name:15} {'#' * int(round(score * 40))} {score:.2f}")

print("\nlearned rules (top of the tree):")
print(export_text(tree, feature_names=FEATURES, max_depth=2))

### Step 4 — The LLM predicts the same rows (10 min)

Now the challenger. Render 10 validation rows as CSV, hand them to the model
with the feature meanings — but **not** the planted rule — and parse 0/1
predictions back. In mock mode a fixed prediction list stands in (8/10 correct
by design; a live model varies run to run).

In [ ]:
sample = df.loc[y_val.sample(n=10, random_state=3).index]
csv_block = sample[FEATURES].to_csv(index=False)
prompt = ("You are a code-review triage model. Predict whether each change passes "
          "review (1) or not (0). Features: lines_changed, files_touched, "
          "author_tenure (years), has_tests (0/1), weekday (0=Mon..4=Fri). "
          "Reply with exactly 10 predictions as a comma-separated list of 0/1, "
          "in row order, no explanation.\n\n" + csv_block)

if course_ai.MOCK:
    raw = course_ai.CANNED_31
    print("(mock mode — fixed prediction list; the live model answers in class)")
else:
    raw = chat(prompt, max_tokens=60)
print("LLM raw reply:", raw)

llm_preds = [int(b) for b in re.findall(r"[01]", raw)][:10]
true = [int(v) for v in sample["review_passes"]]
tree_preds = [int(v) for v in tree.predict(sample[FEATURES])]
assert len(llm_preds) == 10, "expected 10 predictions — tighten the reply format"

print(f"\n{'row':>3} {'true':>4} {'tree':>4} {'llm':>4}")
for i, (t, tp, lp) in enumerate(zip(true, tree_preds, llm_preds)):
    flag = ""
    if tp == t and lp != t:
        flag = "<- tree only"
    elif lp == t and tp != t:
        flag = "<- llm only"
    print(f"{i:>3} {t:>4} {tp:>4} {lp:>4}  {flag}")

tree_10 = sum(p == t for p, t in zip(tree_preds, true))
llm_10 = sum(p == t for p, t in zip(llm_preds, true))
print(f"\non these 10 rows — tree {tree_10}/10 | llm {llm_10}/10")

### Step 5 — The head-to-head (7 min)

Accuracy is one column. Add interpretability, latency, and cost, and the
"which tool?" question starts answering itself — per use case, not in general.

In [ ]:
# latency: time 10 predictions, 100 times, take the mean
t0 = time.perf_counter()
for _ in range(100):
    tree.predict(sample[FEATURES])
tree_ms = (time.perf_counter() - t0) * 1000 / 100

approx_in = len(prompt) // 4                        # rough token estimate
approx_out = 15
PRICE_IN_PER_1K, PRICE_OUT_PER_1K = 0.00015, 0.0006  # illustrative — check your pinned model
llm_cost = approx_in / 1000 * PRICE_IN_PER_1K + approx_out / 1000 * PRICE_OUT_PER_1K

rows = [
    ("accuracy (100-row val)", f"{val_acc:.2f}", "(scored 10 rows only)"),
    ("accuracy (same 10 rows)", f"{tree_10}/10", f"{llm_10}/10"),
    ("interpretable?", "rules you can print", "opaque per call"),
    ("latency, 10 predictions", f"{tree_ms:.2f} ms", "~one API round-trip (seconds)"),
    ("marginal cost, 10 predictions", "$0.000000", f"~${llm_cost:.5f}"),
    ("needs labelled history?", "yes (300 rows here)", "no — but guesses the rule"),
]
w = 30
print(f"{'':{w}}{'decision tree':<28}{'llm'}")
print(f"{'-' * w}{'-' * 28}{'-' * 40}")
for label, a, b in rows:
    print(f"{label:{w}}{a:<28}{b}")

### Step 6 — When each tool wins (5 min)

The tree wins when the problem is **tabular, labelled, and auditable**: you can
print its rules for a reviewer, it is deterministic, and the millionth
prediction costs nothing. The LLM wins when inputs are **unstructured** (read
the diff itself, not five numbers about it) or when there is no labelled history
to train on. Most real systems route: cheap model first, LLM for the weird tail.

Stretch below: the overfitting sweep, live.

In [ ]:
# YOUR CODE (stretch): max_depth sweep — watch train accuracy climb while
# validation accuracy stalls. That gap IS overfitting.
# for d in range(1, 13):
#     t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
#     print(f"depth {d:2}: train {t.score(X_train, y_train):.2f}  val {t.score(X_val, y_val):.2f}")
print("(stretch — uncomment the sweep above and run it)")

## Deliverable

1. Your trained tree: validation accuracy, depth, and the printed rules.
2. The 10-row comparison table (tree vs LLM) and the head-to-head summary.
3. One sentence: for your team's review triage, which tool — and which column of
   the table decided it.

## Reflection

1. The tree recovered most of the planted rule from 300 labelled examples. What
   would the LLM need to match that — and what would it cost per prediction?
2. Which column mattered most for a real deployment: accuracy, interpretability,
   or cost? Does your answer change at 1M predictions/day?
3. What input would break BOTH tools? (A change whose pass/fail depends on the
   diff's *content*, not its metadata.)

## Debrief (instructor-led)

1. Did anyone's live LLM beat the tree on the 10 rows? Small samples flatter —
   what sample size would settle it?
2. Where in your stack is a "tree-shaped" problem currently being solved by an
   LLM — and what would the migration buy you?
3. Bridge to Lab 6.1: this comparison was hand-run once. What would it take to
   run it on every model bump, as a gate?

## Troubleshooting

- **`ModuleNotFoundError: sklearn` / `pandas`** — pre-installed on the VM;
  elsewhere `pip install scikit-learn pandas` in the kernel's environment.
- **`AssertionError: expected 10 predictions` on a live run** — the model added
  prose or a different format. Tighten the prompt ("exactly 10 comma-separated
  0/1, no explanation") or parse more defensively.
- **Live LLM accuracy differs between runs** — expected: the model is
  stochastic. The tree is deterministic; that difference is part of the lesson.
- **A different sklearn version learns a slightly different tree** — the seeds
  keep it close; the comparison still lands. The mock LLM list is fixed.
- **`(mock mode — fixed prediction list)`** — no key visible; the canned list is
  8/10 by design so the head-to-head runs offline.